In [1]:
import re
from pathlib import Path
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableBranch, RunnableLambda, RunnableParallel
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.output_parsers import BaseOutputParser
from langchain_core.retrievers import BaseRetriever
from langchain_openai import ChatOpenAI
from test_exercise import structured_output_reliability

### Task 1

In [14]:
model = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")

In [ ]:
# Classifier
classifier_prompt = ChatPromptTemplate.from_template(
    """Classify the following input into exactly one category and return only the category name. The categories are:
    - technical
    - billing
    - general

    Input: {input}

    Return only the category name."""
)

classifier_chain = (
    classifier_prompt | model | RunnableLambda(lambda x: x.content[0]["text"].strip().lower())
)

# Prompts for each branch
technical_prompt = ChatPromptTemplate.from_template(
    "You are a technical support agent. Help the user solve this technical issue:\n\n{input}"
)

billing_prompt = ChatPromptTemplate.from_template(
    "You are a billing specialist. Help the user with their billing question:\n\n{input}"
)

general_prompt = ChatPromptTemplate.from_template(
    "You are a helpful customer-support agent. Answer the user's question:\n\n{input}"
)

# Add the classification to the input
prepare_input = RunnableLambda(
    lambda x: {
        "input": x["input"],
        "category": classifier_chain.invoke(x),
    }
)

# Branch based on classification
branch = RunnableBranch(
    (lambda x: x["category"] == "technical", technical_prompt),
    (lambda x: x["category"] == "billing", billing_prompt),
    general_prompt,  # default branch
)

# Complete LCEL chain
chain = prepare_input | branch | model

# Example
result = chain.invoke({"input": "My API keeps returning a 401 error."})

print(result.content)

[{'type': 'text', 'text': "Hello! I'd be happy to help you get this sorted out. A **401 Unauthorized** error means that the server recognizes who you are (or is trying to), but your request lacks valid authentication credentials. \n\nTo help me narrow down the cause, could you share a bit more context?\n1. **What API are you using?** (Is it a third-party service like Stripe, OpenAI, or an internal API you are developing?)\n2. **How are you passing the credentials?** (e.g., Bearer token in the header, API key as a query parameter, Basic Auth?)\n3. **Did this start happening suddenly**, or is this a brand-new implementation?\n\nWhile you gather that info, here is a quick checklist of the most common culprits for a 401 error:\n\n### 1. Missing or Malformed Authorization Header\nMake sure you are including the header correctly. For Bearer tokens, it should look like this:\n* `Authorization: Bearer YOUR_API_KEY`\n* *Common mistake:* Forgetting the word `Bearer `, or adding an extra space.\n

### Task 2

In [13]:
# custom parser


class CustomParser(BaseOutputParser[dict]):
    def parse(self, text: str) -> dict:
        result = {}

        for line in text.strip().splitlines():
            if ":" in line:
                key, value = line.split(":", 1)
                result[key.strip().lower()] = value.strip()

        return result

    @property
    def _type(self) -> str:
        return "custom_output_parser"

In [15]:
prompt = ChatPromptTemplate.from_template(
    """Extract the following information from the input and return it in a structured format:
- Name
- Email
Input: {input}

Return the output in the following format:
Name: <name>
Email: <email>

if not value than return simple null place of value.
"""
)


parser = CustomParser()

chain = prompt | model | parser

result = chain.invoke({"input": "My name is John Doe and my email is john.doe@example.com"})

print(result)

{'name': 'John Doe', 'email': 'john.doe@example.com'}


### Task 3

In [ ]:
def custom_retriever(retriever: BaseRetriever, tenant_id: str) -> list:
    def retrieve(query: str) -> list:
        # Use the provided retriever to get documents
        docs = retriever.get_relevant_documents(query)

        # Filter documents based on tenant_id
        filtered_docs = [doc for doc in docs if doc.metadata.get("tenant_id") == tenant_id]

        return filtered_docs

    return retrieve

### Task 4

In [ ]:
class chunk_retriver(BaseRetriever):
    def _get_relevant_documents(self, query: str) -> list:
        # Implement your logic to retrieve documents based on the query
        # For demonstration, returning a static list of documents
        if query:
            return [
                {
                    "content": "Document 1 content",
                    "metadata": {"tenant_id": "tenant_123", "relevance_score": 0.9},
                },
                {
                    "content": "Document 2 content",
                    "metadata": {"tenant_id": "tenant_456", "relevance_score": 0.8},
                },
                {
                    "content": "Document 3 content",
                    "metadata": {"tenant_id": "tenant_123", "relevance_score": 0.95},
                },
            ]


# Replace with actual retriever implementation
retriever = chunk_retriver()


re_ranker = RunnableLambda(
    lambda docs: sorted(
        docs, key=lambda doc: doc["metadata"].get("relevance_score", 0), reverse=True
    )  # set actual re ranker call here
)


prompt = ChatPromptTemplate.from_template(
    """You are a helpful assistant. Based on the following documents, answer the user's question.
Documents:
{documents}

User's question: {question}
Answer:"""
)

chain = retriever | re_ranker | prompt

# Task 6

In [60]:
primary_model = ChatOpenAI(model="gpt-4o-mini")
fallback_model = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")

model = primary_model.with_fallbacks([fallback_model])

In [63]:
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant."),
        ("human", "{question}"),
    ]
)


def mark_primary(response):
    return {
        "response": response,
        "model_used": "gpt-4o-mini",
    }


def mark_fallback(response):
    return {
        "response": response,
        "model_used": "gemini-3.5-flash-lite",
    }


primary_chain = primary_model | RunnableLambda(mark_primary)
fallback_chain = fallback_model | RunnableLambda(mark_fallback)

chain = prompt | primary_chain.with_fallbacks([fallback_chain])


result = chain.invoke({"question": "What is LCEL?"})

print("Model used:", result["model_used"])
print("Response:", result["response"].content)

Model used: gemini-3.5-flash-lite
Response: [{'type': 'text', 'text': '**LCEL** stands for **LangChain Expression Language**. \n\nIt is a declarative framework developed by LangChain (a popular toolkit for building applications with Large Language Models) designed to make it easy to build, compose, and productionize AI chains and agents. \n\nThink of LCEL as a set of plumbing tools that allows you to connect different AI components—like prompts, LLMs, output parsers, and retrieval tools—together like Lego bricks.\n\nHere is a breakdown of why LCEL is important and how it works:\n\n### 1. The Core Idea: The Pipe Operator (`|`)\nThe defining feature of LCEL is its heavy use of the Unix pipe operator (`|`). In LCEL, the output of one component is automatically fed as the input to the next component.\n\nHere is a simple example in Python:\n```python\nfrom langchain_core.output_parsers import StrOutputParser\nfrom langchain_core.prompts import ChatPromptTemplate\nfrom langchain_openai impor

### Task 7

In [67]:
def log_rendered_prompt(step_name: str):
    """
    Creates a Runnable that logs the exact rendered prompt
    before it is sent to the model.
    """

    def logger(prompt_value):
        print(f"\n{'=' * 100}")
        print(f"PROMPT - {step_name}")
        print(f"{'=' * 100}")

        # For chat prompts, show each rendered message
        for message in prompt_value.to_messages():
            print(f"[{message.type.upper()}]")
            print(message.content)

        print(f"{'=' * 100}\n")

        # Important: return the original value so the chain continues
        return prompt_value

    return RunnableLambda(logger)


# Step 1 prompt
prompt_step_1 = ChatPromptTemplate.from_messages(
    [
        ("system", "You are an expert text analyzer."),
        ("human", "Analyze this text and identify the main topic:\n\n{text}"),
    ]
)


# Step 2 prompt
prompt_step_2 = ChatPromptTemplate.from_messages(
    [
        ("system", "You are an expert summarizer."),
        ("human", "Create a short summary of this analysis:\n\n{analysis}"),
    ]
)


# Step 1
step_1 = prompt_step_1 | log_rendered_prompt("Step 1 - Analysis") | model


# Step 2
step_2 = prompt_step_2 | log_rendered_prompt("Step 2 - Summary") | model


# Multi-step chain
def run_chain(text: str):
    analysis = step_1.invoke({"text": text})

    summary = step_2.invoke({"analysis": analysis.content})

    return summary


# Run
result = run_chain(
    "LangChain provides tools for building applications using large language models."
)


print(f"{'=' * 100}\n")
print("Final response:")
print(result.content)


PROMPT - Step 1 - Analysis
[SYSTEM]
You are an expert text analyzer.
[HUMAN]
Analyze this text and identify the main topic:

LangChain provides tools for building applications using large language models.


PROMPT - Step 2 - Summary
[SYSTEM]
You are an expert summarizer.
[HUMAN]
Create a short summary of this analysis:

[{'type': 'text', 'text': 'Based on the text provided, the main topic is **LangChain** (specifically, its function as a toolset for building applications powered by large language models).', 'extras': {'signature': 'El4KXAERTTIPIjabSM+SUHzu7Rqc2efip6tCGKE3ky4GPR3GWfY1oMQg9Xmx3sYyhIIiLgsOMbHhBlz+I371H3ozMasDyr76058DluU4N8HdcElZZZ/BHbNufuOrORBr'}}]


Final response:
[{'type': 'text', 'text': 'Based on the provided analysis, the text is about **LangChain**, focusing on its role as a toolset for developing applications powered by large language models.', 'extras': {'signature': 'El4KXAERTTIPOYAp1qbFKVXGPlWOqOME5n/fTBKh0PsD0QVQwC8RTAy4IUlF9SuS5VcQogzVG1y82q3cMFjk6wLUgX/PCjX

### Task 8

In [69]:
# Summarization prompt
summary_prompt = ChatPromptTemplate.from_template(
    """
    Summarize the following text in 2-3 sentences:

    {text}
    """
)

# Classification prompt
classification_prompt = ChatPromptTemplate.from_template(
    """
    Classify the following text into exactly one category:
    technical, billing, general

    Text:
    {text}

    Return only the category.
    """
)


# Two parallel sub-chains
parallel_chain = RunnableParallel(
    summary=summary_prompt | model,
    classification=classification_prompt | model,
)


# Run both chains in parallel
result = await parallel_chain.ainvoke(
    {
        "text": """
    The customer cannot log into the application because
    their password reset email never arrives.
    """
    }
)


# Access results
print("SUMMARY:")
print(result["summary"].content)

print("\nCLASSIFICATION:")
print(result["classification"].content)

Direct use of automatic function calling (AFC) in AsyncModels.generate_content is not recommended. Instead, we recommend to use AFC in AsyncChat.send_message. Similarly, direct use of AFC in AsyncModels.generate_content_stream is not recommended. Instead, we recommend to use AFC in AsyncChat.send_message_stream.


SUMMARY:
[{'type': 'text', 'text': 'The customer is unable to log into the application because they are not receiving their password reset email. This issue is preventing them from accessing their account.', 'extras': {'signature': 'El4KXAERTTIPk+B4thcrzJspsGlqsy3wURjGZ/ZFkvhbH6GYu7XVf0grWyL8buF7ufCFh7cSeMJC0f1pC8oTfsqA1sYeniSThB40/p/Qmu76OAJxaip/KENlBxfPmtxr'}}]

CLASSIFICATION:
[{'type': 'text', 'text': 'technical', 'extras': {'signature': 'El4KXAERTTIP6zWAG/6NS7BC1jezyPvEb/wcZCmJH6OmVo/SC5OTiO94FuebacuTZU5lAK20jTCQ8YUVbHYe8fjeYHVeHcyEUyBpdYBwKQF0StoMelU9oyYGRfZGRgLI'}}]


### Task 9

In [3]:
structured_output_reliability()

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Rate limit hit. Retrying in 31 seconds...

STRUCTURED OUTPUT COMPARISON

Golden set: 13 cases

Module 4 Custom Harness
Valid:       13/13
Reliability: 100.0%
Correct:     12/13
Accuracy:    92.31%

LangChain PydanticOutputParser
Valid:       13/13
Reliability: 100.0%
Correct:     12/13
Accuracy:    92.31%



### Task 10

In [77]:
SOURCE_DIR = Path("prompts/")
OUTPUT_FILE = Path("migrated_prompts.py")


def migrate_prompt(path: Path) -> str:
    text = path.read_text()

    # Convert Jinja variables:
    # {{ user_query }} -> {user_query}
    text = re.sub(r"\{\{\s*(\w+)\s*\}\}", r"{\1}", text)

    # Treat the first part as system prompt and the rest as human prompt
    parts = text.split("\n\n", 1)

    if len(parts) == 2:
        system_prompt, human_prompt = parts
    else:
        system_prompt = "You are a helpful assistant."
        human_prompt = text

    return f"""ChatPromptTemplate.from_messages([
    ("system", {system_prompt!r}),
    ("human", {human_prompt!r}),
])"""


def migrate():
    prompts = []

    for path in SOURCE_DIR.glob("*.md"):
        name = path.stem

        template_code = migrate_prompt(path)

        prompts.append(f"{name.upper()} = {template_code}")

    output = """from langchain_core.prompts import ChatPromptTemplate

""" + "\n\n\n".join(prompts)
    OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
    OUTPUT_FILE.write_text(output)

    print(f"Migrated {len(prompts)} prompts.")
    print(f"Saved to: {OUTPUT_FILE}")

In [78]:
migrate()

Migrated 2 prompts.
Saved to: migrated_prompts.py
